# 2. Parse filings and create Silver chunks

This notebook reads raw documents from the **Bronze** table and writes cleaned, indexed text chunks to the **Silver** Delta table, `financial_db.filings.sec_rag_silver`. Run the ingestion notebook first and configure Databricks Connect with read/write access to these tables.

### What the cells do

1. **Configure input and output tables.** Set the catalog/schema and identify the Bronze source and Silver destination.

In [ ]:
from databricks.connect import DatabricksSession
import os
from bs4 import BeautifulSoup
import regex as re
from pyspark.sql.functions import col, concat_ws
from dotenv import load_dotenv

load_dotenv()

catalog_name = "financial_db"
schema_name = "filings"

BRONZE_TABLE = f"{catalog_name}.{schema_name}.sec_rag_bronze"
TABLE = f"{catalog_name}.{schema_name}.sec_rag_silver"

2. **Connect and prepare storage.** Start a Databricks Connect session and create the Silver table if it does not exist.

In [ ]:
os.environ.pop("DATABRICKS_METADATA_SERVICE_URL", None)
os.environ["DATABRICKS_AUTH_TYPE"] = "pat"
os.environ["DATABRICKS_SERVERLESS_COMPUTE_ID"] = "auto"

spark = DatabricksSession.builder.getOrCreate()

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {TABLE} (
    chunk_id STRING,
    ticker STRING,
    cik STRING,
    date STRING,
    accession_number STRING,
    url STRING,
    chunk_index INT,
    chunk_text STRING,
    chunk_length INT,
    section_chars INT
)
USING DELTA
TBLPROPERTIES (
    'delta.minReaderVersion' = '1',
    'delta.minWriterVersion' = '2'
)
''')

3. **Clean, extract, and chunk.** Remove HTML scripts, styles, and tables; normalize whitespace; try to extract Item 1A (Risk Factors) through Item 1B or Item 2; and fall back to a bounded prefix of the document if a sufficiently long risk-factor section is not found. Split the resulting text into 1,000-character chunks with 200-character overlap, preserving filing metadata and chunk positions.

In [ ]:
def parse(raw: str):
    if not raw:
        return ""

    soup = BeautifulSoup(raw, 'html.parser')
    for elem in soup(['script', 'style', 'table']):
        elem.decompose()
    text = soup.get_text(separator=' ')
    cleaned_text = re.sub(r'\s+', ' ', text).strip()
    pattern = re.compile(r'(item\s+1a[\.\:]?\s*risk\s+factors)(.*?)(item\s+(?:1b|2)[\.\:]?\s*)', re.IGNORECASE | re.DOTALL)

    match = pattern.search(cleaned_text)
    if match:
        extracted = match.group(2).strip()
        if len(extracted) > 500:
            return extracted

    return cleaned_text[:50000]

def make_chunks(text: str, chunk_size: int = 1000, overlap: int = 200):
    chunks = []
    ini = 0
    counter = 0
    n = len(text)

    while ini < n:
        fin = min(ini + 1000, n)
        chunk = text[ini:fin].strip()
        chunks.append({
            "index": counter,
            "text": chunk,
            "length": len(chunk)
        })
        counter += 1
        ini = ini + chunk_size - overlap

    return chunks

def process(raw):
    res = []

    item_1a = parse(raw)
    chunks = make_chunks(item_1a)
    total_chars = len(item_1a)

    for c in chunks:
        c["total_chars"] = total_chars
            
    return chunks

def get_silver():
    df_bronze = spark.sql(f'''
        SELECT ticker, cik, date, accession_number, url, raw
        FROM {BRONZE_TABLE}
        WHERE accession_number NOT IN (SELECT DISTINCT accession_number FROM {TABLE})
    ''')

    new_chunks = df_bronze.count()
    if new_chunks > 0:
        df_pd = df_bronze.toPandas()
        df_pd["chunks"] = df_pd["raw"].apply(process)
        df_pd = df_pd.explode("chunks").reset_index(drop=True)
        df = spark.createDataFrame(df_pd)
        df = df.select(
                concat_ws("_", col("accession_number"), col("chunks.index")).alias("chunk_id"),
                col("ticker"),
                col("cik"),
                col("date"),
                col("accession_number"),
                col("url"),
                col("chunks.index").alias("chunk_index"),
                col("chunks.text").alias("chunk_text"),
                col("chunks.length").alias("chunk_length"),
                col("chunks.total_chars").alias("section_chars")
            )
        df.write.format("delta").mode("append").saveAsTable(TABLE)

get_silver()

4. **Inspect output.** Query the chunk table to review filing dates, chunk positions, lengths, and source-section size.

In [ ]:
spark.sql(f'''SELECT ticker, date, chunk_index, chunk_length, section_chars FROM {TABLE}
ORDER BY chunk_index''')